[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/37_uncertainty_and_calibration/02_platt_and_isotonic.ipynb)

# Bài 02 — Calibration hậu kiểm: Platt, isotonic, temperature scaling

**🎯 Mục tiêu**
- Sửa xác suất **sau khi** train, trên một **tập calibration** riêng — không đụng tới mô hình.
- **Platt scaling** (hồi quy logistic trên điểm số) và **isotonic regression** (thuật toán PAV) from scratch; đối chiếu scikit-learn.
- Chọn phương pháp theo **cỡ tập calibration**; `CalibratedClassifierCV`.
- **Temperature scaling** cho bài toán nhiều lớp.
- Điều chỉnh xác suất khi **tỉ lệ nền** (base rate) thay đổi — ví dụ sau khi cân bằng lại dữ liệu.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/37_uncertainty_and_calibration

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import load_digits
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore", category=UserWarning)

raw = make_customer_churn(30000, seed=0)
X = pd.DataFrame({
    "tenure": raw.tenure_months, "monthly": raw.monthly_charges, "total": raw.total_charges,
    "support_calls": raw.support_calls, "month_to_month": (raw.contract == "month-to-month").astype(int),
    "two_year": (raw.contract == "two_year").astype(int), "fiber": (raw.internet_service == "fiber").astype(int),
    "ewallet": (raw.payment_method == "e-wallet").astype(int), "senior": raw.senior.astype(int), "n_products": raw.n_products,
}).astype(float)
true_logit = (-1.6 + 1.3 * X.month_to_month - 0.5 * X.two_year - 0.035 * X.tenure + 0.018 * (X.monthly - 70) + 0.38 * X.support_calls
              + 0.45 * X.fiber + 0.35 * X.ewallet + 0.3 * X.senior - 0.2 * (X.n_products - 1))
p_true = 1 / (1 + np.exp(-true_logit.values))
y = raw.churn.values
idx_train, idx_rest = train_test_split(np.arange(len(X)), test_size=0.4, random_state=0)
idx_cal, idx_test = train_test_split(idx_rest, test_size=0.5, random_state=0)
y_cal, y_test, pt_test = y[idx_cal], y[idx_test], p_true[idx_test]


def ece(p, y, n_bins=10):
    bins = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    return sum((bins == b).mean() * abs(y[bins == b].mean() - p[bins == b].mean()) for b in range(n_bins) if (bins == b).any())


def report(name, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {"": name, "ROC-AUC": roc_auc_score(y_test, p), "ECE": ece(p, y_test), "Brier": brier_score_loss(y_test, p),
            "log-loss": log_loss(y_test, p), "MSE với p thật": np.mean((p - pt_test) ** 2)}

## 1. Ý tưởng: học một ánh xạ từ điểm số sang xác suất

Mô hình đã train cho điểm số $s(x)$ (xác suất chưa chuẩn, decision function...). Calibration hậu kiểm học một hàm **đơn điệu tăng** $g$ sao cho $g(s(x))$ calibrated, trên **tập calibration** — dữ liệu **chưa dùng để train** (dùng chính tập train thì điểm số đã quá khớp, ánh xạ học được sẽ sai). Vì $g$ đơn điệu, **thứ tự** không đổi → AUC gần như giữ nguyên.

Hai mô hình cần sửa (từ bài 01): Naive Bayes (quá tự tin do giả định độc lập sai) và MLP overfit.

In [ ]:
nb = GaussianNB().fit(X.iloc[idx_train], y[idx_train])
mlp = make_pipeline(StandardScaler(), MLPClassifier((128, 128), alpha=0, max_iter=500, random_state=0)).fit(X.iloc[idx_train[:3000]], y[idx_train[:3000]])
scores = {name: (m.predict_proba(X.iloc[idx_cal])[:, 1], m.predict_proba(X.iloc[idx_test])[:, 1]) for name, m in [("Naive Bayes", nb), ("MLP", mlp)]}

## 2. Platt scaling

Platt (1999): $g(s) = \sigma(a\cdot z + b)$ với $z$ là logit của điểm số — tức một **hồi quy logistic 1 biến** trên tập calibration. Hai tham số: ít dữ liệu cũng ước lượng được, nhưng chỉ sửa được sai lệch có **dạng sigmoid** (quá/kém tự tin đối xứng).

In [ ]:
def to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_platt(s_cal, y_cal, iters=50):
    """Newton–Raphson cho hồi quy logistic 1 biến: p = σ(a·logit(s) + b)."""
    z = to_logit(s_cal)
    a, b = 0.0, 0.0                                                    # bắt đầu từ p = 0.5: Hessian xác định tốt; loss lồi
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(a * z + b, -30, 30)))
        grad = np.array([np.sum((p - y_cal) * z), np.sum(p - y_cal)])
        w = p * (1 - p)
        hess = np.array([[np.sum(w * z * z), np.sum(w * z)], [np.sum(w * z), np.sum(w)]]) + 1e-9 * np.eye(2)
        a, b = np.array([a, b]) - np.linalg.solve(hess, grad)
    return lambda s: 1 / (1 + np.exp(-(a * to_logit(s) + b))), (a, b)


platt_nb, (a, b) = fit_platt(scores["Naive Bayes"][0], y_cal)
ref = LogisticRegression(C=1e10).fit(to_logit(scores["Naive Bayes"][0]).reshape(-1, 1), y_cal)
print(f"Platt cho Naive Bayes: a = {a:.4f}, b = {b:.4f} | sklearn: a = {ref.coef_[0, 0]:.4f}, b = {ref.intercept_[0]:.4f}")

$a < 1$: logit của Naive Bayes bị **phóng đại** — Platt "co" lại, đúng chẩn đoán quá tự tin ở bài 01.

## 3. Isotonic regression — thuật toán PAV

Isotonic regression (Zadrozny & Elkan, 2002) tìm hàm **bậc thang đơn điệu tăng bất kỳ** khớp nhất (bình phương tối thiểu) với nhãn khi sắp xếp theo điểm số. Thuật toán **Pool Adjacent Violators** (PAV): đi từ trái sang phải; mỗi khi khối mới có trung bình **nhỏ hơn** khối trước (vi phạm đơn điệu), **gộp** chúng lại và lấy trung bình; lặp tới khi đơn điệu.

In [ ]:
def pav(y_sorted):
    """Isotonic regression trên dãy đã sắp theo điểm số. Trả về giá trị khớp cho từng phần tử."""
    blocks = []                                                        # mỗi khối: [tổng, số phần tử]
    for v in y_sorted:
        blocks.append([float(v), 1])
        while len(blocks) > 1 and blocks[-2][0] / blocks[-2][1] > blocks[-1][0] / blocks[-1][1]:
            total, count = blocks.pop()
            blocks[-1][0] += total
            blocks[-1][1] += count
    return np.concatenate([np.full(c, t / c) for t, c in blocks])


def fit_isotonic(s_cal, y_cal):
    order = np.argsort(s_cal, kind="mergesort")
    xs, fitted = s_cal[order], pav(y_cal[order])
    return lambda s: np.interp(s, xs, fitted)                          # nội suy, kẹp ở hai đầu


s_cal, s_test = scores["Naive Bayes"]
mine = fit_isotonic(s_cal, y_cal)(s_test)
sk = IsotonicRegression(out_of_bounds="clip").fit(s_cal, y_cal).predict(s_test)
print(f"sai lệch so với sklearn IsotonicRegression: {np.abs(mine - sk).max():.2e} | số mức khác nhau: {len(np.unique(mine))}")

(Sai lệch rất nhỏ còn lại đến từ cách xử lý điểm số trùng nhau.) Isotonic linh hoạt hơn Platt — sửa được mọi dạng sai lệch đơn điệu — nhưng nhiều "tham số" hơn → cần nhiều dữ liệu calibration hơn, và tạo ra xác suất dạng bậc thang (có thể làm **giảm nhẹ AUC** do gộp các điểm số khác nhau thành cùng một mức).

## 4. So sánh trên tập test

In [ ]:
rows = []
for name, (s_cal, s_test) in scores.items():
    platt, _ = fit_platt(s_cal, y_cal)
    iso = fit_isotonic(s_cal, y_cal)
    rows += [report(f"{name} — gốc", s_test), report(f"{name} — Platt", platt(s_test)), report(f"{name} — isotonic", iso(s_test))]
rows.append(report("xác suất thật", pt_test))
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, (name, (s_cal, s_test)) in zip(axes, scores.items()):
    order = np.argsort(s_test)
    ax.plot(s_test[order], fit_platt(s_cal, y_cal)[0](s_test[order]), label="Platt")
    ax.plot(s_test[order], fit_isotonic(s_cal, y_cal)(s_test[order]), label="isotonic")
    ax.plot([0, 1], [0, 1], "k:", label="không đổi")
    ax.set_xlabel(f"xác suất gốc của {name}")
    ax.set_ylabel("xác suất sau calibration")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Cả hai phương pháp giảm mạnh ECE, Brier, log-loss và đưa xác suất gần $p^*$ hơn nhiều; ROC-AUC gần như không đổi (thứ tự được giữ). Đường ánh xạ cho thấy chẩn đoán: xác suất gốc gần 0 hoặc 1 bị kéo vào giữa — mô hình gốc quá tự tin.

## 5. Bao nhiêu dữ liệu calibration là đủ?

Calibrate Naive Bayes với $n$ mẫu calibration (10 lần chọn mẫu), đo MSE so với xác suất thật trên tập test:

In [ ]:
s_cal_full, s_test = scores["Naive Bayes"]
results = []
for n in [50, 200, 1000, 6000]:
    errs = {"Platt": [], "isotonic": []}
    for seed in range(10):
        sub = np.random.default_rng(seed).choice(len(idx_cal), n, replace=False)
        errs["Platt"].append(np.mean((fit_platt(s_cal_full[sub], y_cal[sub])[0](s_test) - pt_test) ** 2))
        errs["isotonic"].append(np.mean((fit_isotonic(s_cal_full[sub], y_cal[sub])(s_test) - pt_test) ** 2))
    results.append({"n calibration": n, **{f"MSE với p thật — {k}": np.mean(v) for k, v in errs.items()}})
print(pd.DataFrame(results).round(4).to_string(index=False))

Với rất ít dữ liệu, Platt (2 tham số) ổn định hơn; khi đủ dữ liệu (vài trăm trở lên), isotonic linh hoạt hơn và thắng — nhưng ở đây Platt bị giới hạn vì sai lệch của Naive Bayes không hoàn toàn có dạng sigmoid. Quy tắc thực hành phổ biến: Platt khi tập calibration nhỏ (< ~1 000), isotonic khi lớn.

### `CalibratedClassifierCV`: không cần tách tập riêng

`CalibratedClassifierCV(model, method=..., cv=5)` train mô hình trên 4/5 dữ liệu, học ánh xạ calibration trên 1/5 còn lại, lặp 5 lần và **trung bình** 5 mô hình đã calibrate — tận dụng hết dữ liệu.

In [ ]:
cal_cv = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(X.iloc[idx_train], y[idx_train])
print(pd.DataFrame([report("Naive Bayes — CalibratedClassifierCV(isotonic, cv=5)", cal_cv.predict_proba(X.iloc[idx_test])[:, 1])]).round(4).to_string(index=False))

## 6. Nhiều lớp: temperature scaling

Với softmax nhiều lớp, cách đơn giản và rất hiệu quả cho mạng nơ-ron (Guo et al., 2017): chia **mọi** logit cho một **nhiệt độ** $T > 0$ duy nhất, chọn $T$ cực tiểu log-loss trên tập calibration. $T > 1$ làm phân phối "mềm" đi (giảm tự tin); argmax — và accuracy — không đổi.

Thí nghiệm: chữ số viết tay 8×8, **20% nhãn bị gán sai** (mô phỏng nhãn nhiễu thực tế); MLP không regularization học thuộc cả nhãn sai → rất quá tự tin.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16
rng = np.random.default_rng(0)
y_noisy = yd.copy()
flip = rng.random(len(yd)) < 0.2
y_noisy[flip] = rng.integers(0, 10, flip.sum())
d_train, d_rest = train_test_split(np.arange(len(Xd)), test_size=0.5, random_state=0, stratify=yd)
d_cal, d_test = train_test_split(d_rest, test_size=0.5, random_state=0, stratify=yd[d_rest])
digits_mlp = MLPClassifier((256,), alpha=0, max_iter=3000, random_state=0).fit(Xd[d_train], y_noisy[d_train])


def multiclass_ece(P, y, n_bins=15):
    conf, pred = P.max(1), P.argmax(1)
    bins = np.clip((conf * n_bins).astype(int), 0, n_bins - 1)
    return sum((bins == b).mean() * abs((pred[bins == b] == y[bins == b]).mean() - conf[bins == b].mean()) for b in range(n_bins) if (bins == b).any())


def with_temperature(log_probs, T):
    z = log_probs / T
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


L_cal = np.log(np.clip(digits_mlp.predict_proba(Xd[d_cal]), 1e-15, 1))
L_test = np.log(np.clip(digits_mlp.predict_proba(Xd[d_test]), 1e-15, 1))
temperatures = np.linspace(0.5, 10, 191)
best_T = temperatures[np.argmin([log_loss(y_noisy[d_cal], with_temperature(L_cal, T), labels=range(10)) for T in temperatures])]
P_before, P_after = with_temperature(L_test, 1.0), with_temperature(L_test, best_T)
yt = y_noisy[d_test]
print(f"accuracy train = {(digits_mlp.predict(Xd[d_train]) == y_noisy[d_train]).mean():.2f} (học thuộc cả nhãn sai) | test = {(P_before.argmax(1) == yt).mean():.3f}")
print(f"T tối ưu = {best_T:.2f}")
for name, P in [("trước", P_before), ("sau temperature scaling", P_after)]:
    print(f"{name:24s}: độ tự tin TB = {P.max(1).mean():.3f} | ECE = {multiclass_ece(P, yt):.3f} | log-loss = {log_loss(yt, P, labels=range(10)):.3f} | "
          f"accuracy = {(P.argmax(1) == yt).mean():.3f}")

$T \gg 1$: mô hình tự tin gần 90% trong khi chỉ đúng khoảng 2/3. Một tham số duy nhất giảm log-loss hơn một nửa và giảm mạnh ECE mà accuracy giữ nguyên — nhưng không sửa hết (ECE vẫn đáng kể): một nhiệt độ chung không thể sửa sai lệch khác nhau giữa các lớp/vùng; các mở rộng: vector/matrix scaling, Dirichlet calibration. Cũng lưu ý: calibration hậu kiểm **không** sửa được việc mô hình đã học thuộc nhãn sai — chỉ sửa độ tự tin.

## 7. Khi tỉ lệ nền thay đổi

Chương 19: khi dữ liệu mất cân bằng, người ta hay **undersample** lớp đa số để train. Xác suất của mô hình train trên dữ liệu 50/50 phản ánh tỉ lệ **50/50**, không phải tỉ lệ thật → bị thổi phồng. Nếu chỉ tỉ lệ nền thay đổi ($P(x\mid y)$ giữ nguyên), điều chỉnh bằng định lý Bayes (Saerens et al., 2002; Elkan, 2001):

$$p_{\text{mới}} = \frac{p\cdot\frac{\pi_1'}{\pi_1}}{p\cdot\frac{\pi_1'}{\pi_1} + (1 - p)\cdot\frac{\pi_0'}{\pi_0}}$$

($\pi$: tỉ lệ lớp khi train, $\pi'$: tỉ lệ lớp khi triển khai).

In [ ]:
pos, neg = idx_train[y[idx_train] == 1], idx_train[y[idx_train] == 0]
balanced = np.r_[pos, np.random.default_rng(0).choice(neg, len(pos), replace=False)]
lr_bal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X.iloc[balanced], y[balanced])
p_bal = lr_bal.predict_proba(X.iloc[idx_test])[:, 1]
pi_train, pi_deploy = 0.5, y[idx_train].mean()
ratio1, ratio0 = pi_deploy / pi_train, (1 - pi_deploy) / (1 - pi_train)
p_adj = p_bal * ratio1 / (p_bal * ratio1 + (1 - p_bal) * ratio0)
print(f"tỉ lệ rời bỏ thật trên test = {y_test.mean():.3f}")
print(pd.DataFrame([report("train 50/50 — xác suất gốc", p_bal), report("train 50/50 — điều chỉnh tỉ lệ nền", p_adj)]).round(4).to_string(index=False))
print(f"xác suất dự đoán trung bình: gốc = {p_bal.mean():.3f} | sau điều chỉnh = {p_adj.mean():.3f}")

Undersampling làm xác suất trung bình bị đẩy lên (gần 0.5) — ECE lớn dù AUC không đổi. Công thức điều chỉnh tỉ lệ nền khôi phục calibration mà không cần dữ liệu calibration. Cùng ý tưởng dùng khi triển khai ở thị trường có tỉ lệ khác (prior shift — chương 5) nếu ước lượng được tỉ lệ mới.

## 🧠 Kiểm tra nhanh

1. Vì sao phải calibrate trên tập riêng, không phải tập train?
2. Platt scaling học gì? Khi nào phù hợp?
3. Mô tả thuật toán PAV. Isotonic khác Platt thế nào về độ linh hoạt và lượng dữ liệu cần?
4. Temperature scaling hoạt động thế nào? Vì sao accuracy không đổi?
5. Train trên dữ liệu đã cân bằng 50/50 ảnh hưởng xác suất thế nào? Sửa ra sao?

## 📌 Tóm tắt

- Calibration hậu kiểm: học ánh xạ đơn điệu điểm số → xác suất trên tập calibration riêng; AUC gần như giữ nguyên.
- Platt: sigmoid 2 tham số, ổn định khi ít dữ liệu; isotonic (PAV): bậc thang đơn điệu tuỳ ý, cần nhiều dữ liệu hơn; `CalibratedClassifierCV` tận dụng hết dữ liệu.
- Nhiều lớp: temperature scaling — một tham số, hiệu quả với mạng nơ-ron, không sửa hết.
- Tỉ lệ nền thay đổi (undersampling, prior shift): điều chỉnh theo Bayes.

## 📚 Tài liệu tham khảo

- Platt (1999) — *Probabilistic Outputs for Support Vector Machines and Comparisons to Regularized Likelihood Methods*
- Zadrozny & Elkan (2002) — *Transforming Classifier Scores into Accurate Multiclass Probability Estimates*
- Guo et al. (2017) — *On Calibration of Modern Neural Networks* (temperature scaling)
- Saerens, Latinne & Decaestecker (2002) — *Adjusting the Outputs of a Classifier to New a Priori Probabilities*
- Kull et al. (2019) — *Beyond temperature scaling: Obtaining well-calibrated multiclass probabilities with Dirichlet calibration*